# 05 · The tool-call round trip on three vendor wire formats
Level L1 · Part 1 · core · ~20 min · builds on 04

## Goal
Declare one tool, let the model ask for it, run it in your code, and send the result back in the Gemini and
Claude formats (both recorded and replayed here). For each vendor, say which part of the reply is the request and
which part you send back. The OpenAI format was not run live in this course (no key): its shapes come from the
OpenAI Responses API docs, and the lesson shows them without making a call.

## Why it matters
Without a tool contract, your code must guess from free text whether to look up an order. With one, the model
returns a structured request (a name and arguments) and your code answers it.

| Vendor | The model asks with | You send the result back as |
|---|---|---|
| Gemini | a `function_call` part in its reply | a `function_response` part in a user turn |
| Claude | a `tool_use` block (with an `id`) in its reply | a `tool_result` block that repeats the `tool_use_id` |
| OpenAI Responses (not run live; from the docs) | a `function_call` item (with a `call_id`) in `output` | a `function_call_output` item with the same `call_id` |

A **turn** is one message in the conversation list. A `call_id` names one OpenAI request; the `function_call_output`
item is your answer, with the same `call_id`.

**What it makes easy:** one schema serves every vendor, and your loop does not parse text. **When you don't need it:**
a fixed lookup your code always runs.

## The idea
A **tool call** is a request from the model, not an action. The model writes a tool name and arguments that match
a **JSON Schema** you gave it. Your code runs the function and sends the result back, tied to the request by an
id. Then the model writes its answer from that result. The round trip has two model calls: one for the request,
one for the answer. The model may ask for several tools in one reply (**parallel tool calls**). **`tool_choice`**
tells it whether it may, must or must not call a tool.

Analogy: a restaurant order. The waiter (the model) writes a ticket with a dish number and a table number. The
kitchen (your code) makes the dish, and the dish goes back with the same ticket number. The waiter then serves it.

## Picture
```mermaid
sequenceDiagram
  participant App as Your code
  participant M as Model
  App->>M: messages + tool schemas + tool_choice
  M-->>App: tool request(s): name, arguments, id
  App->>App: run each function (validate first)
  App->>M: the results, each tied to its id, plus the model's own turn
  M-->>App: final answer
```

## Step by step
**Step 1 · one function and one schema (no call).** `lookup_order` reads a tiny order table made for this lesson
(fictional, not the real CRM). The JSON Schema describes its one argument. All three vendors accept this schema,
only the wrapper around it differs. `run_tool` finds the function by name in a registry, so adding a tool does not
change the loop.

In [1]:
import json

ORDERS = {  # fictional rows for this lesson; the real CRM comes in a later lesson
    "1093": {"status": "in transit", "days_late": 3},
    "2207": {"status": "delivered", "days_late": 0},
}


def lookup_order(order_id: str) -> dict:
    """Return the status of one order, or an error the model can read and act on."""
    return ORDERS.get(order_id, {"error": f"no order with id {order_id}"})


from pydantic import BaseModel, ConfigDict, Field


class OrderQuery(BaseModel):
    """The arguments of lookup_order. Pydantic writes the JSON Schema and checks each call."""
    model_config = ConfigDict(extra="forbid")  # no unknown keys: the schema gets additionalProperties: false
    order_id: str = Field(description="The order number the customer wrote, digits only, such as 1093.")


ORDER_SCHEMA = OrderQuery.model_json_schema()   # the one schema all three vendors receive
TOOLS = {"lookup_order": lookup_order}          # the registry: name -> function
SCHEMAS = {"lookup_order": OrderQuery}


def run_tool(name: str, args: dict) -> dict:
    checked = SCHEMAS[name].model_validate(args).model_dump()  # refuses bad arguments before the function runs
    return TOOLS[name](**checked)


print(run_tool("lookup_order", {"order_id": "1093"}))

{'status': 'in transit', 'days_late': 3}


**Step 2 · the clients (the setup).** Each SDK gets its key from `.env`, never typed here. While a recording
replays, a placeholder is used and no key is sent. `llm_replay` is a course helper, not part of the SDKs: it saves
every HTTP call the SDKs make, so later runs replay the saved answers. A round trip needs two calls. The course
limit of five is a cost budget for this notebook's recorded calls. Gemini uses two and Claude two. The OpenAI part
makes no call, because no key was available.

In [2]:
import sys; sys.path.insert(0, "..")
import llm_replay  # course only: records and replays model calls (not part of the SDK)
llm_replay.start("05_tool-call-round-trip")

import anthropic  # the official Anthropic SDK; in your project: anthropic.Anthropic(api_key=...)
from google import genai  # the official Google SDK; in your project: genai.Client(api_key=...)
from google.genai import types
from tut_env import env  # course only: reads settings from the .env file (not part of the SDK)

NO_KEY = "replay"  # course only: while replaying, the recording answers and this placeholder is never sent
REPLAYING = llm_replay.is_replaying()
gemini = genai.Client(vertexai=True, api_key=NO_KEY if REPLAYING else env("GOOGLE_CLOUD_API_KEY"),
                      http_options=types.HttpOptions(timeout=30_000))
claude = anthropic.Anthropic(api_key=NO_KEY if REPLAYING else env("ANTHROPIC_API_KEY"), timeout=30)
MODEL_GEMINI, MODEL_CLAUDE, MODEL_OPENAI = "gemini-3.1-flash-lite", "claude-haiku-5-5", "gpt-6-luna"
PROMPT = "Where are my orders 1093 and 2207? Check both, then answer in two sentences."
print("clients ready · replaying:", REPLAYING)

LLM mode: replay (05_tool-call-round-trip.yaml)


clients ready · replaying: True


**Step 3 · Gemini: a `function_call` part, then a `function_response` part.** The tool is a `FunctionDeclaration`.
`mode="AUTO"` lets the model decide. Call 1 returns the model's turn, which can hold several `function_call`
parts. Your code keeps that turn as it is, and adds one user turn with a `function_response` per call. Call 2
returns the answer. Gemini matches each `function_response` to its `function_call` by name and order; this lesson
sends no id. The SDK does not run the function for you here (automatic calling needs Python functions), so the
loop is visible.

In [3]:
gemini_config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=[types.FunctionDeclaration(
        name="lookup_order", description="Look up the status of one order by its number.",
        parameters_json_schema=ORDER_SCHEMA)])],
    tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode="AUTO")),
    temperature=0,
)
history = [types.Content(role="user", parts=[types.Part(text=PROMPT)])]

r1 = gemini.models.generate_content(model=MODEL_GEMINI, contents=history, config=gemini_config)   # call 1
calls = [p.function_call for p in r1.candidates[0].content.parts if p.function_call]
for c in calls:
    print("Gemini asks:", c.name, dict(c.args))
history.append(r1.candidates[0].content)  # the model's own turn, unchanged
history.append(types.Content(role="user", parts=[
    types.Part.from_function_response(name=c.name, response=run_tool(c.name, dict(c.args))) for c in calls]))

r2 = gemini.models.generate_content(model=MODEL_GEMINI, contents=history, config=gemini_config)   # call 2
print("Gemini answer:", r2.text)

Gemini asks: lookup_order {'order_id': '1093'}
Gemini asks: lookup_order {'order_id': '2207'}
Gemini answer: Order 1093 is currently in transit and is three days late. Order 2207 has been successfully delivered.


**Step 4 · Claude: `tool_use` blocks, then `tool_result` blocks in one user message.** The schema goes under
`input_schema`. `stop_reason == "tool_use"` says the model stopped to ask. Each `tool_use` block has an `id`. Your
reply puts every `tool_result` in one user message, and each repeats its `tool_use_id`. `max_tokens` is required.
Claude Haiku 5.5 accepts only the default temperature, so the call does not set one. The cell ends the recorder:
the last model call of this notebook is the Claude answer.

In [4]:
claude_tools = [{"name": "lookup_order", "description": "Look up the status of one order by its number.",
                 "input_schema": ORDER_SCHEMA}]
messages = [{"role": "user", "content": PROMPT}]

m1 = claude.messages.create(model=MODEL_CLAUDE, max_tokens=1024, tools=claude_tools,
                            tool_choice={"type": "auto"}, messages=messages)   # call 1
asks = [b for b in m1.content if b.type == "tool_use"]
print("Claude stop reason:", m1.stop_reason)
for b in asks:
    print("Claude asks:", b.name, b.input, "· id", b.id)
messages.append({"role": "assistant", "content": m1.content})  # the model's own turn, unchanged
messages.append({"role": "user", "content": [
    {"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(run_tool(b.name, b.input))} for b in asks]})

m2 = claude.messages.create(model=MODEL_CLAUDE, max_tokens=1024, tools=claude_tools,
                            tool_choice={"type": "auto"}, messages=messages)   # call 2
print("Claude answer:", next(b.text for b in m2.content if b.type == "text"))
llm_replay.stop()  # last model call of this notebook; nothing below calls a model

Claude stop reason: tool_use
Claude asks: lookup_order {'order_id': '1093'} · id toolu_01Ft69QkhgurrbNhe8s3ydHq
Claude asks: lookup_order {'order_id': '2207'} · id toolu_01B5pHWuUgs7b8S4VFnzsrk1
Claude answer: Order 1093 is in transit and 3 days late. Order 2207 has been delivered and arrived on time.


**Step 5 · OpenAI Responses: shape only, no call.** This part was not run live in this course (no key), so the
cell sends no request. Its shapes come from the OpenAI function-calling guide
(https://developers.openai.com/api/docs/guides/function-calling). The Responses API takes `tools` with the schema
under `parameters`. The reply's `output` holds the requests as items with a `call_id` and `arguments`, a JSON
string. Your next request adds those items and one `function_call_output` per call, with the same `call_id`. The
cell prints the request it would send, built from the same tools and settings as a live call, and two illustrative
items with placeholder ids.

In [5]:
openai_tools = [{"type": "function", "name": "lookup_order", "strict": True,
                 "description": "Look up the status of one order by its number.", "parameters": ORDER_SCHEMA}]
openai_request = {"model": MODEL_OPENAI, "input": PROMPT, "tools": openai_tools,
                  "tool_choice": "auto", "parallel_tool_calls": True}
print("Request this cell would send (not sent):")
print(json.dumps(openai_request, indent=2))

# Illustrative shapes from the docs, not model output. The ids are placeholders.
example_call = {"type": "function_call", "call_id": "call_example", "name": "lookup_order",
                "arguments": '{"order_id": "1093"}'}
example_output = {"type": "function_call_output", "call_id": "call_example",
                  "output": '{"status": "in transit", "days_late": 3}'}
print("illustrative shape, not model output · function_call item:", example_call)
print("illustrative shape, not model output · function_call_output item:", example_output)

# The openai SDK's own types accept these shapes (validation only; nothing is sent).
from openai.types.responses import ResponseFunctionToolCall  # the SDK's type for a function_call item
from openai.types.responses.response_input_item import FunctionCallOutput  # and for a function_call_output item
ResponseFunctionToolCall.model_validate(example_call)  # raises an error if the shape is wrong
FunctionCallOutput.model_validate(example_output)
print("the openai SDK types accept both shapes")

Request this cell would send (not sent):
{
  "model": "gpt-6-luna",
  "input": "Where are my orders 1093 and 2207? Check both, then answer in two sentences.",
  "tools": [
    {
      "type": "function",
      "name": "lookup_order",
      "strict": true,
      "description": "Look up the status of one order by its number.",
      "parameters": {
        "additionalProperties": false,
        "description": "The arguments of lookup_order. Pydantic writes the JSON Schema and checks each call.",
        "properties": {
          "order_id": {
            "description": "The order number the customer wrote, digits only, such as 1093.",
            "title": "Order Id",
            "type": "string"
          }
        },
        "required": [
          "order_id"
        ],
        "title": "OrderQuery",
        "type": "object"
      }
    }
  ],
  "tool_choice": "auto",
  "parallel_tool_calls": true
}
illustrative shape, not model output · function_call item: {'type': 'function_call', 'ca

the openai SDK types accept both shapes


## Under the hood
The SDK call is one step. Below is the minimal round trip for one Claude reply: run the tools it asked for, send
the results back in the same conversation, and read the next reply. It uses the recorded `m1` and `m2`, so no new
call is made. There is no loop and no cap here. Lesson 06 adds the loop, the step cap and the error handling.

In [6]:
def one_round_trip(reply, send):
    """Run the tools one reply asked for, send the results back, and return the model's next reply."""
    asks = [b for b in reply.content if b.type == "tool_use"]
    messages = [{"role": "user", "content": PROMPT},
                {"role": "assistant", "content": reply.content},
                {"role": "user", "content": [{"type": "tool_result", "tool_use_id": b.id,
                                              "content": json.dumps(run_tool(b.name, b.input))} for b in asks]}]
    return send(messages)


next_reply = one_round_trip(m1, lambda messages: m2)  # send() stands in for claude.messages.create; m2 is recorded
print("answer:", next(b.text for b in next_reply.content if b.type == "text"))

answer: Order 1093 is in transit and 3 days late. Order 2207 has been delivered and arrived on time.


## At work
- Teams keep one registry and one JSON Schema per tool, then adapt it to each vendor at the edge. A loop hard-coded to one vendor breaks when a field changes → lesson 20 (vendor SDKs or a hand loop).
- Every tool request is an instruction from the model. Teams validate the arguments and run a policy check before a write runs, with an approval step for refunds → lesson 09 (safety) and lesson 14 (approval gate).
- Parallel calls cut waiting time, but each call needs a timeout, a retry for transient errors and a cap on steps → lesson 06 (loop caps) and lesson 32 (reliability).

## Common mistakes
1. **Trusting the arguments.** The model's arguments are a request, not a checked value. Without a check, a wrong
   key fails deep inside your function with an error that tells the model nothing. Check against the schema first,
   as `run_tool` does. Lesson 07 covers error text the model can act on.

In [7]:
run_tool("lookup_order", {"id": "1093"})   # the model sent "id", but the schema says "order_id"

ValidationError: 2 validation errors for OrderQuery
order_id
  Field required [type=missing, input_value={'id': '1093'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
id
  Extra inputs are not permitted [type=extra_forbidden, input_value='1093', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/extra_forbidden

The error names two problems: `order_id` is missing, and `id` is an extra field. One wrong key counts as one
missing field plus one extra field.

2. **Dropping the model's turn or the ids.** The second request must hold the model's own request and a result
   for each request id. Claude's docs say each `tool_use` id needs its `tool_result` in the next user message
   (https://platform.claude.com/docs/en/agents-and-tools/tool-use/handle-tool-calls). The OpenAI guide says each
   `function_call_output` must carry the matching `call_id`.
3. **Splitting parallel results.** Claude expects every `tool_result` for one turn in one user message.

## Try it
`tool_choice` means something slightly different on each vendor. Write `tool_choice_for(vendor, choice)` that
returns the setting each vendor expects, for `choice` in `"auto"`, `"required"`, `"none"` and `"tool"` (force
`lookup_order`). Use the names in the table: `"gemini"`, `"claude"`, `"openai"`. No model call is needed.

| Choice | Gemini (`function_calling_config`) | Claude (`tool_choice`) | OpenAI (`tool_choice`) |
|---|---|---|---|
| may call a tool | `mode="AUTO"` | `{"type": "auto"}` | `"auto"` |
| must call a tool | `mode="ANY"` | `{"type": "any"}` | `"required"` |
| must not call | `mode="NONE"` | `{"type": "none"}` | `"none"` |
| must call one named tool | `mode="ANY"` + `allowed_function_names` | `{"type": "tool", "name": ...}` | `{"type": "function", "name": ...}` |

This table is for this lesson's code. Check each vendor's docs before you rely on a value.

In [8]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [9]:
def tool_choice_for(vendor: str, choice: str, tool: str = "lookup_order"):
    table = {
        "gemini": {"auto": {"mode": "AUTO"}, "required": {"mode": "ANY"}, "none": {"mode": "NONE"},
                   "tool": {"mode": "ANY", "allowed_function_names": [tool]}},
        "claude": {"auto": {"type": "auto"}, "required": {"type": "any"}, "none": {"type": "none"},
                   "tool": {"type": "tool", "name": tool}},
        "openai": {"auto": "auto", "required": "required", "none": "none",
                   "tool": {"type": "function", "name": tool}},
    }
    return table[vendor][choice]


for vendor in ["gemini", "claude", "openai"]:
    print(vendor, "forced tool:", tool_choice_for(vendor, "tool"))

gemini forced tool: {'mode': 'ANY', 'allowed_function_names': ['lookup_order']}
claude forced tool: {'type': 'tool', 'name': 'lookup_order'}
openai forced tool: {'type': 'function', 'name': 'lookup_order'}


## Recap
- A tool call is a request: the model names a function and arguments; your code runs it and returns the result with the same id.
- The round trip makes two model calls: the request, then the answer written from the result.
- Gemini (`function_call`, `function_response` parts) and Claude (`tool_use`, `tool_result` blocks) ran here from recordings.
- OpenAI Responses (`function_call`, `function_call_output` items; not run live; from the docs). All three use the same JSON Schema.
- `tool_choice` sets whether the model may, must or must not call a tool; each vendor spells the same choice differently.

**Interview:** "Walk me through what the model returns when it wants a tool, and what your code sends back." Strong answer: "It returns a request with a name, arguments and an id. My code validates the arguments, runs the function and sends the result with the same id in the vendor's format. Parallel requests come back together and go back together."